## Wikipedia API Testing

Search funcitonality

In [6]:
import os
import requests

def test_wikipedia_search(query: str, limit: int = 5):
    url = "https://en.wikipedia.org/w/rest.php/v1/search/page"
    
    # Wikimedia requires a descriptive User-Agent
    headers = {
        "User-Agent": "TriviaQA (dzureca7@gmail.com)"
    }
    
    params = {
        "q": query,
        "limit": limit
    }
    
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code == 200:
        data = response.json()
        print(f"--- Search Results for '{query}' ---")
        for page in data.get("pages", []):
            print(f"Title: {page['title']}")
            print(f"Description: {page.get('description', 'No description')}")
            print("-" * 20)
    else:
        print(f"Error: {response.status_code}")
        print(response.text)

# Run the test
test_wikipedia_search("Quantum computing")

--- Search Results for 'Quantum computing' ---
Title: Quantum computing
Description: Computer hardware technology that uses quantum mechanics
--------------------
Title: Superconducting quantum computing
Description: Quantum computing implementation
--------------------
Title: Timeline of quantum computing and communication
Description: 
--------------------
Title: Post-quantum cryptography
Description: Cryptography secured against quantum computers
--------------------
Title: Trapped-ion quantum computer
Description: Proposed quantum computer implementation
--------------------


Unstructured parsing logic

In [14]:
from unstructured.partition.html import partition_html

def parse_wikipedia_html(html_content: str):
    print("Parsing HTML with unstructured...")
    
    # We pass the raw string directly to the 'text' parameter
    elements = partition_html(text=html_content)
    
    # unstructured returns a list of Element objects. 
    # Let's see how it categorized the first few pieces of the article:
    print("\n--- Element Categorization ---")
    for el in elements[:5]:
        print(f"[{type(el).__name__}]: {str(el)[:50]}...")
        
    # Now, let's filter out the junk and rebuild a clean document.
    # We only want the meaty parts: Titles, Narrative Text, and Lists.
    desired_categories = ["Title", "NarrativeText", "ListItem"]
    
    clean_paragraphs = []
    for el in elements:
        if type(el).__name__ in desired_categories:
            # We can clean up some common Wikipedia artifacts here if needed
            text = str(el).strip()
            if text:
                clean_paragraphs.append(text)
                
    # Join the clean elements with double newlines for excellent LLM readability
    final_markdown_ish_text = "\n\n".join(clean_paragraphs)
    
    return final_markdown_ish_text

Inspect Page functionality

In [15]:
import os
import requests

def test_wikipedia_inspect(title: str):
    formatted_title = title.replace(" ", "_")
    
    # Pointing directly to the HTML endpoint instead of /bare
    url = f"https://en.wikipedia.org/w/rest.php/v1/page/{formatted_title}/html"
    
    headers = {
        "User-Agent": "MyLLMWikiBot/1.0 (your_email@example.com)"
    }
    
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        # Because the response is raw HTML, we use .text instead of .json()
        html_content = response.text
        
        print(f"--- Raw HTML for '{title}' ---")
        
        markdown_content = parse_wikipedia_html(html_content)
        print(markdown_content)
        return markdown_content
        
    else:
        print(f"Error: {response.status_code}")
        print(response.text)

# Run the test
test_wikipedia_inspect("Quantum computing")

--- Raw HTML for 'Quantum computing' ---
Parsing HTML with unstructured...

--- Element Categorization ---
[NarrativeText]: Computer hardware technology that uses quantum mec...
[NarrativeText]: A quantum computer is a (real or theoretical) comp...
[NarrativeText]: The basic unit of information in quantum computing...
[NarrativeText]: Quantum computers are not yet practical for real-w...
[Title]: History...
Computer hardware technology that uses quantum mechanics

A quantum computer is a (real or theoretical) computer that exploits superposed and entangled states. Quantum computers can be viewed as sampling from quantum systems. These systems evolve in ways that operate on an enormous number of possibilities simultaneously, though they remain subject to strict computational constraints. By contrast, ordinary ("classical") computers operate according to deterministic rules. (A classical computer can, in principle, be replicated by a classical mechanical device, with only a simple multip

'Computer hardware technology that uses quantum mechanics\n\nA quantum computer is a (real or theoretical) computer that exploits superposed and entangled states. Quantum computers can be viewed as sampling from quantum systems. These systems evolve in ways that operate on an enormous number of possibilities simultaneously, though they remain subject to strict computational constraints. By contrast, ordinary ("classical") computers operate according to deterministic rules. (A classical computer can, in principle, be replicated by a classical mechanical device, with only a simple multiple of time cost. On the other hand (it is believed), a quantum computer would require exponentially more time and energy to be simulated classically.) It is widely believed that a quantum computer could perform some calculations exponentially faster than any classical computer. For example, a large-scale quantum computer could break some widely used public-key cryptographic schemes and aid physicists in p